# Notebook 09 — Dense-lead upgrade if the shared stores support it

One of the main remaining weaknesses in the current paper is that the flagship analysis uses only
the sparse lead set `24/72/120/168 h`. This notebook tries to upgrade the strongest **all-years**
run to a denser common lead ladder, while keeping the same codebase and the same runner functions.

Workflow:
1. Inspect the all-years alignment audit.
2. Infer a common dense lead set if the audit exposes one.
3. Rerun the deterministic atlas, growth, and blocking sections at the dense leads.
4. Save a manifest and a short readiness note.

Outputs go to:

`notebooks/outputs/flagship_96plus/denselead_upgrade/`

In [ ]:

from pathlib import Path
import sys
import json
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def detect_bundle_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd.parent.parent,
    ]
    for base in candidates:
        if (base / "src" / "flagship_predictability").exists() and (base / "examples" / "default_settings.py").exists():
            return base
        if (base / "flagship_predictability_nextpass" / "src" / "flagship_predictability").exists():
            return base / "flagship_predictability_nextpass"
    raise RuntimeError(
        "Could not detect the bundle root. Run from inside the project tree, "
        "or edit detect_bundle_root() with the correct path."
    )

BUNDLE_ROOT = detect_bundle_root()
SRC_ROOT = BUNDLE_ROOT / "src"
NOTEBOOK_ROOT = BUNDLE_ROOT / "notebooks"
OUTPUT_ROOT = NOTEBOOK_ROOT / "outputs" / "flagship_96plus"

for p in [BUNDLE_ROOT, SRC_ROOT]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Bundle root:", BUNDLE_ROOT)
print("Output root:", OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from examples.default_settings import SETTINGS as BASE_SETTINGS

from flagship_predictability import run_dataset_audit, run_deterministic_atlas, run_growth_diagnostics, run_blocking_verification


In [ ]:
def set_attr_any(obj, names, value):
    """
    Set the first writable attribute from `names`.
    Skip read-only properties and continue to the next candidate.
    """
    last_err = None

    for name in names:
        if hasattr(obj, name):
            try:
                setattr(obj, name, value)
                return name
            except (AttributeError, TypeError) as e:
                last_err = e
                continue

    for name in names:
        try:
            setattr(obj, name, value)
            return name
        except (AttributeError, TypeError) as e:
            last_err = e
            continue

    if hasattr(obj, "__dict__"):
        obj.__dict__[names[0]] = value
        return names[0]

    raise AttributeError(
        f"Could not set any of {names} on {type(obj).__name__}. "
        f"Last error: {last_err}"
    )


def get_attr_any(obj, names, default=None):
    for name in names:
        if hasattr(obj, name):
            return getattr(obj, name)
    return default


def configure_window(settings, start, end):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    set_attr_any(settings, ["analysis_start", "window_start", "start_time", "start", "verification_start"], start)
    set_attr_any(settings, ["analysis_end", "window_end", "end_time", "end", "verification_end"], end)
    return settings


def configure_leads(settings, lead_hours):
    lead_hours = [int(x) for x in lead_hours]
    lead_strings = [f"{x}h" for x in lead_hours]
    lead_tds = [pd.Timedelta(hours=x) for x in lead_hours]

    set_attr_any(settings, ["lead_hours", "forecast_hours"], lead_hours)
    set_attr_any(settings, ["leads", "lead_times", "flagship_leads"], lead_strings)
    set_attr_any(settings, ["lead_timedeltas", "lead_td", "forecast_timedeltas"], lead_tds)
    return settings


def configure_regimes(settings, n_regimes=4, n_components=8, n_boot=400, block_length=5):
    set_attr_any(settings, ["n_regimes", "regime_count"], int(n_regimes))
    set_attr_any(settings, ["n_eof", "n_components", "regime_n_components"], int(n_components))
    set_attr_any(settings, ["bootstrap_samples", "n_boot", "n_bootstrap"], int(n_boot))
    set_attr_any(settings, ["bootstrap_block_length", "block_length"], int(block_length))
    return settings


def configure_blocking(settings, threshold=0.1):
    set_attr_any(settings, ["blocking_threshold"], float(threshold))
    return settings


def configure_growth(settings, lag_hours=12):
    set_attr_any(settings, ["growth_lag_hours", "lag_hours", "trajectory_lag_hours"], int(lag_hours))
    return settings

def td_hours(series):
    return (pd.to_timedelta(series).dt.total_seconds() / 3600.0)

def td_days(series):
    return (pd.to_timedelta(series).dt.total_seconds() / 86400.0)

def season_name(ts):
    m = pd.Timestamp(ts).month
    return {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
            6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}[m]

def bootstrap_ci(values, n_boot=2000, alpha=0.05, seed=42):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan, np.nan, np.nan
    if vals.size == 1:
        return float(vals[0]), float(vals[0]), float(vals[0])
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        sample = rng.choice(vals, size=vals.size, replace=True)
        means[i] = sample.mean()
    lo, hi = np.quantile(means, [alpha / 2.0, 1 - alpha / 2.0])
    return float(vals.mean()), float(lo), float(hi)

def safe_read_csv(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Could not read {path}: {e}")
        return pd.DataFrame()

def set_pub_style():
    plt.rcParams.update({
        "figure.dpi": 170,
        "savefig.dpi": 300,
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.titlesize": 13,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "lines.linewidth": 2.0,
        "lines.markersize": 5,
    })


MULTIWINDOW_ROOT = OUTPUT_ROOT / "multiwindow_runs"
DENSE_ROOT = OUTPUT_ROOT / "denselead_upgrade"
DENSE_ROOT.mkdir(parents=True, exist_ok=True)

ALL_YEARS_TAG = "all_2018_2022"
CANDIDATE_DENSE_LEADS_H = [24, 48, 72, 96, 120, 144, 168]
DEFAULT_BLOCKING_THRESHOLD = 0.1
DEFAULT_GROWTH_LAG_H = 12

manifest_path = MULTIWINDOW_ROOT / "multiwindow_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError("Run Notebook 08 first so the multiwindow manifest exists.")

manifest = pd.read_csv(manifest_path)
manifest.head()


In [ ]:

def infer_common_leads_from_audit(audit_df, candidate_hours):
    if audit_df.empty:
        return []
    cols = list(audit_df.columns)

    lead_col = next((c for c in cols if "lead" in c.lower()), None)
    if lead_col is None:
        return []

    model_col = next((c for c in cols if c.lower() in {"model", "dataset", "forecast_dataset", "source"}), None)
    variable_col = next((c for c in cols if c.lower() in {"variable", "var"}), None)
    status_col = next((c for c in cols if c.lower() == "status"), None)

    work = audit_df.copy()
    if status_col is not None:
        work = work[work[status_col].astype(str).str.lower().eq("ok")]

    # convert whatever the lead representation is into hours
    try:
        work["lead_h"] = td_hours(work[lead_col]).round().astype("Int64")
    except Exception:
        try:
            work["lead_h"] = pd.to_numeric(work[lead_col], errors="coerce").round().astype("Int64")
        except Exception:
            return []

    work = work.dropna(subset=["lead_h"])
    work = work[work["lead_h"].isin(candidate_hours)]

    if variable_col is not None:
        work = work[work[variable_col].astype(str).isin(["z500", "t850", "u850", "v850"])]

    if work.empty:
        return []

    if model_col is None:
        return sorted(work["lead_h"].astype(int).unique().tolist())

    groups = []
    for _, grp in work.groupby(model_col):
        groups.append(set(grp["lead_h"].astype(int).unique().tolist()))
    if not groups:
        return []
    common = set.intersection(*groups)
    return sorted(common)

row = manifest[manifest["tag"] == ALL_YEARS_TAG].iloc[0]
audit_candidates = [c for c in manifest.columns if c.startswith("audit__")]
audit_paths = [Path(row[c]) for c in audit_candidates if pd.notna(row.get(c, None))]
audit_df = pd.DataFrame()
for p in audit_paths:
    if p.exists() and p.suffix == ".csv":
        tmp = safe_read_csv(p)
        if not tmp.empty:
            audit_df = tmp
            break

dense_leads_h = infer_common_leads_from_audit(audit_df, CANDIDATE_DENSE_LEADS_H)
if not dense_leads_h:
    print("Could not infer a common dense lead set automatically; using the candidate list as a best-effort default.")
    dense_leads_h = CANDIDATE_DENSE_LEADS_H.copy()

dense_leads_h


In [ ]:

dense_summary = {
    "all_years_tag": ALL_YEARS_TAG,
    "requested_candidate_leads_h": CANDIDATE_DENSE_LEADS_H,
    "selected_dense_leads_h": dense_leads_h,
    "n_selected_dense_leads": len(dense_leads_h),
}

summary_path = DENSE_ROOT / "denselead_summary.json"
summary_path.write_text(json.dumps(dense_summary, indent=2), encoding="utf-8")
summary_path


In [ ]:

# If the dense lead ladder is not meaningfully denser than the flagship one, we still write the summary
# but skip the reruns.
if len(dense_leads_h) <= 4:
    print("Dense lead set is not larger than the flagship sparse set; skipping reruns.")
else:
    settings = deepcopy(BASE_SETTINGS)
    settings = configure_window(settings, "2018-01-01", "2022-12-31")
    settings = configure_leads(settings, dense_leads_h)
    settings = configure_blocking(settings, DEFAULT_BLOCKING_THRESHOLD)
    settings = configure_growth(settings, DEFAULT_GROWTH_LAG_H)

    rerun_manifest = {}

    det_out = run_deterministic_atlas(settings, output_root=DENSE_ROOT)
    rerun_manifest["deterministic"] = {k: str(v) for k, v in det_out.items()}

    try:
        gr_out = run_growth_diagnostics(settings, output_root=DENSE_ROOT)
        rerun_manifest["growth"] = {k: str(v) for k, v in gr_out.items()}
    except Exception as e:
        rerun_manifest["growth"] = {"status": f"error: {e}"}

    try:
        bl_out = run_blocking_verification(settings, output_root=DENSE_ROOT)
        rerun_manifest["blocking"] = {k: str(v) for k, v in bl_out.items()}
    except Exception as e:
        rerun_manifest["blocking"] = {"status": f"error: {e}"}

    (DENSE_ROOT / "denselead_rerun_manifest.json").write_text(json.dumps(rerun_manifest, indent=2), encoding="utf-8")
    print("Dense-lead reruns complete.")


In [ ]:

print((DENSE_ROOT / "denselead_summary.json").read_text())

det_dense = safe_read_csv(DENSE_ROOT / "deterministic" / "deterministic_summary.csv")
display(det_dense.head())

if not det_dense.empty and "lead" in det_dense.columns:
    display(det_dense.assign(lead_h=td_hours(det_dense["lead"])).groupby(["model", "variable"])["lead_h"].nunique().unstack())
